# 🔧 Notebook 2: Prétraitement du Texte

## Projet Anti-Spam Intelligent - BMSecurity

**Objectif:** Nettoyer et préparer les textes des emails pour l'entraînement du modèle.

### Ce qu'on va faire:
1. Charger les données
2. Nettoyer le texte (minuscules, supprimer ponctuation)
3. Supprimer les doublons et valeurs manquantes
4. Tokenisation (découper en mots)
5. Supprimer les stopwords (mots courants sans valeur)
6. Stemming (réduire les mots à leur racine)
7. Vectorisation (convertir texte en nombres)
8. Sauvegarder les données prétraitées

---

## Étape 1: Importer les bibliothèques

On importe les outils pour le traitement de texte (NLP).

In [ ]:
# Bibliothèques de base
import pandas as pd
import numpy as np
import re  # Pour les expressions régulières

# Bibliothèques NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# Vectorisation
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Télécharger les ressources NLTK nécessaires
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

print("✅ Bibliothèques importées!")

## Étape 2: Charger les données

In [ ]:
# Charger le dataset
df = pd.read_csv('../DataSet_Emails/DataSet_Emails.csv')

# Garder seulement les colonnes dont on a besoin
df = df[['text', 'label', 'label_text']].copy()

print(f"Dataset chargé: {len(df)} emails")
df.head()

## Étape 3: Nettoyer les données

On supprime les doublons et les lignes avec du texte vide.

In [ ]:
# Taille avant nettoyage
print(f"Avant nettoyage: {len(df)} lignes")

# Supprimer les doublons
df = df.drop_duplicates(subset=['text'])
print(f"Après suppression doublons: {len(df)} lignes")

# Supprimer les lignes avec texte vide ou manquant
df = df.dropna(subset=['text'])
df = df[df['text'].str.strip() != '']
print(f"Après suppression textes vides: {len(df)} lignes")

## Étape 4: Créer la fonction de prétraitement

On crée une fonction qui va:
1. Convertir en minuscules
2. Supprimer la ponctuation et caractères spéciaux
3. Tokeniser (découper en mots)
4. Supprimer les stopwords
5. Appliquer le stemming

In [ ]:
# Initialiser le stemmer et les stopwords
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Fonction pour prétraiter un texte:
    1. Minuscules
    2. Supprimer ponctuation
    3. Tokeniser
    4. Supprimer stopwords
    5. Stemming
    """
    # Convertir en string si nécessaire
    text = str(text)
    
    # 1. Convertir en minuscules
    text = text.lower()
    
    # 2. Supprimer ponctuation et caractères spéciaux
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 3. Tokeniser (découper en mots)
    tokens = word_tokenize(text)
    
    # 4. Supprimer les stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # 5. Appliquer le stemming
    tokens = [stemmer.stem(word) for word in tokens]
    
    # Rejoindre les mots
    return ' '.join(tokens)

print("✅ Fonction de prétraitement créée!")

## Étape 5: Tester la fonction sur un exemple

Voyons comment la fonction transforme un texte.

In [ ]:
# Test avec un exemple
exemple = "Hello! This is a TEST message with some SPECIAL characters: @#$%"

print("Texte original:")
print(exemple)
print()
print("Texte après prétraitement:")
print(preprocess_text(exemple))

## Étape 6: Appliquer le prétraitement à tout le dataset

⚠️ Cette étape peut prendre quelques minutes selon la taille du dataset.

In [ ]:
# Appliquer la fonction à tous les emails
print("Prétraitement en cours...")
df['text_clean'] = df['text'].apply(preprocess_text)
print("✅ Prétraitement terminé!")

# Voir le résultat
df[['text', 'text_clean', 'label_text']].head()

## Étape 7: Vectorisation avec TF-IDF

On convertit les textes en vecteurs numériques que le modèle pourra comprendre.

**TF-IDF** = Term Frequency - Inverse Document Frequency
- Donne plus d'importance aux mots rares et discriminants
- Donne moins d'importance aux mots très fréquents

In [ ]:
# Créer le vectoriseur TF-IDF
# max_features=5000 = on garde les 5000 mots les plus importants
tfidf = TfidfVectorizer(max_features=5000)

# Appliquer la vectorisation
X = tfidf.fit_transform(df['text_clean'])
y = df['label']

print(f"Forme de X (features): {X.shape}")
print(f"Forme de y (labels): {y.shape}")
print()
print(f"Nombre de mots dans le vocabulaire: {len(tfidf.vocabulary_)}")

## Étape 8: Sauvegarder les données prétraitées

On sauvegarde pour pouvoir les utiliser dans le prochain notebook.

In [ ]:
import pickle
import os

# Créer le dossier models s'il n'existe pas
os.makedirs('../models', exist_ok=True)

# Sauvegarder le vectoriseur TF-IDF
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Sauvegarder les données prétraitées
df.to_csv('../models/data_preprocessed.csv', index=False)

# Sauvegarder X et y
import scipy.sparse
scipy.sparse.save_npz('../models/X_tfidf.npz', X)
np.save('../models/y_labels.npy', y.values)

print("✅ Données sauvegardées dans le dossier 'models':")

## ✅ Résumé du prétraitement

**Ce qu'on a fait:**
1. ✅ Chargé les données
2. ✅ Supprimé les doublons et valeurs manquantes
3. ✅ Nettoyé le texte (minuscules, ponctuation)
4. ✅ Tokenisé les textes
5. ✅ Supprimé les stopwords
6. ✅ Appliqué le stemming
7. ✅ Vectorisé avec TF-IDF
8. ✅ Sauvegardé les données

**Prochaine étape:** Notebook 3 - Entraînement des modèles